# IPCV Assignment — Emotion Detection in Social Media Images

# By Chinelo Lydia Nweke

## Data Source
###  Download FER2013 from Kaggle:
###    https://www.kaggle.com/datasets/msambare/fer2013

## Dataset Loading & Exploration

### Loading Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### 1. CONFIGURATION

In [2]:
DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

In [4]:
from pathlib import Path

downloads = Path.home() / "Downloads"
candidates = []
for p in downloads.rglob("*"):
    if p.is_dir():
        if (p / "train").is_dir() and (p / "test").is_dir():
            candidates.append(p)

print("Downloads:", downloads)
if candidates:
    print("\nFound candidate folders (set SRC_ROOT to one of these):")
    for c in sorted(set(candidates)):
        print(" -", c)
else:
    print("\nNo candidate archive folder found under Downloads.")
    print("You can:")
    print(" - Move the downloaded archive to Downloads and unzip it, or")
    print(" - Copy the folder path and paste it into SRC_ROOT (see next cell).")

Downloads: C:\Users\nweke\Downloads

Found candidate folders (set SRC_ROOT to one of these):
 - C:\Users\nweke\Downloads\archive (1)


In [5]:
from pathlib import Path
import shutil

# set SRC_ROOT to the folder found by the search
SRC_ROOT = Path(r"C:\Users\nweke\Downloads\archive (1)")
DEST_ROOT = Path("data")                      # matches notebook's DATA_DIR
CLASSES = ["angry","disgust","fear","happy","neutral","sad","surprise"]
SPLITS = ("train","test")
OVERWRITE = True  # set False to keep existing folders

if not SRC_ROOT.exists():
    raise FileNotFoundError(f"Source not found: {SRC_ROOT}")

DEST_ROOT.mkdir(parents=True, exist_ok=True)
for split in SPLITS:
    src_split = SRC_ROOT / split
    dst_split = DEST_ROOT / split
    dst_split.mkdir(parents=True, exist_ok=True)
    if not src_split.exists():
        print(f"Missing source split: {src_split}")
        continue
    print(f"\nCopying split: {split}")
    for cls in CLASSES:
        src_cls = src_split / cls
        dst_cls = dst_split / cls
        if not src_cls.exists():
            print(f"  missing: {src_cls}")
            continue
        if dst_cls.exists():
            if OVERWRITE:
                shutil.rmtree(dst_cls)
            else:
                print(f"  exists, skipping: {dst_cls}")
                continue
        shutil.copytree(src_cls, dst_cls)
        n = sum(1 for _ in dst_cls.rglob('*') if _.is_file())
        print(f"  copied {n:4d} files -> {dst_cls}")

# Print counts
print("\nCounts in data/:")
for split in SPLITS:
    p = DEST_ROOT / split
    print(f" {split}:")
    if p.exists():
        for d in sorted([d for d in p.iterdir() if d.is_dir()], key=lambda x: x.name):
            n = sum(1 for f in (p / d.name).rglob('*') if f.is_file())
            print(f"   {d.name:10s} - {n} files")
    else:
        print(f"   missing {p}")


Copying split: train
  copied 3995 files -> data\train\angry
  copied  436 files -> data\train\disgust
  copied 4097 files -> data\train\fear
  copied 7215 files -> data\train\happy
  copied 4965 files -> data\train\neutral
  copied 4830 files -> data\train\sad
  copied 3171 files -> data\train\surprise

Copying split: test
  copied  958 files -> data\test\angry
  copied  111 files -> data\test\disgust
  copied 1024 files -> data\test\fear
  copied 1774 files -> data\test\happy
  copied 1233 files -> data\test\neutral
  copied 1247 files -> data\test\sad
  copied  831 files -> data\test\surprise

Counts in data/:
 train:
   angry      - 3995 files
   disgust    - 436 files
   fear       - 4097 files
   happy      - 7215 files
   neutral    - 4965 files
   sad        - 4830 files
   surprise   - 3171 files
 test:
   angry      - 958 files
   disgust    - 111 files
   fear       - 1024 files
   happy      - 1774 files
   neutral    - 1233 files
   sad        - 1247 files
   surprise   -

## EDA for image data

### Class counts (train/test)

In [ ]:
from pathlib import Path
DATA_DIR = Path("data")
for split in ("train","test"):
    p = DATA_DIR / split
    if not p.exists():
        print(f"Missing {p}")
        continue
    print(f"\n{split}:")
    for d in sorted([d for d in p.iterdir() if d.is_dir()], key=lambda x: x.name):
        n = sum(1 for f in (p / d.name).rglob('*') if f.is_file())
        print(f"  {d.name:12s} - {n} files")

### Find corrupted/unreadable images

In [ ]:
from PIL import Image, UnidentifiedImageError
from pathlib import Path

bad = []
for f in (Path("data")/"train").rglob("*"):
    if not f.is_file(): continue
    try:
        with Image.open(f) as im:
            im.verify()
    except Exception:
        bad.append(str(f))
print("Corrupted/unreadable images:", len(bad))
# optionally list them
bad[:20]

### Image size / aspect ratio distribution

In [ ]:
from PIL import Image
from collections import Counter
sizes = Counter()
for f in (Path("data")/"train").rglob("*"):
    if f.is_file():
        with Image.open(f) as im:
            sizes[im.size] += 1
sizes.most_common(10)

### Per-channel mean/std (approx on full training set)

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

sum_ = np.zeros(3)
sum_sq = np.zeros(3)
count = 0
for f in (Path("data")/"train").rglob("*"):
    if not f.is_file(): continue
    with Image.open(f) as im:
        arr = np.array(im.convert("RGB"), dtype=np.float32)/255.0
        sum_ += arr.mean(axis=(0,1))
        sum_sq += (arr**2).mean(axis=(0,1))
        count += 1
mean = sum_/count
std = np.sqrt(sum_sq/count - mean**2)
print("approx mean:", mean, "approx std:", std)

### Show sample grid per class (quick visual spot-check)

In [ ]:
import matplotlib.pyplot as plt
from random import sample
from PIL import Image
from pathlib import Path

train = Path("data")/"train"
classes = [d.name for d in train.iterdir() if d.is_dir()]
n_per = 5
fig, axes = plt.subplots(len(classes), n_per, figsize=(n_per*2, len(classes)*2))
for i, cls in enumerate(sorted(classes)):
    files = list((train/cls).glob("*"))
    files = sample(files, min(n_per, len(files)))
    for j in range(n_per):
        ax = axes[i,j] if len(classes)>1 else axes[j]
        if j < len(files):
            img = Image.open(files[j]).convert("RGB")
            ax.imshow(img)
            ax.set_title(cls if j==0 else "")
        ax.axis("off")
plt.tight_layout()

### Detect exact duplicate files (hash)

In [ ]:
import hashlib
from pathlib import Path
hashes = {}
dups = []
for f in (Path("data")/"train").rglob("*"):
    if not f.is_file(): continue
    h = hashlib.md5(f.read_bytes()).hexdigest()
    if h in hashes:
        dups.append((f, hashes[h]))
    else:
        hashes[h] = f
len(dups), dups[:5]

### Make a CSV manifest (path, split, class) for downstream pipelines

In [ ]:
import csv
from pathlib import Path

out = Path("data/manifest.csv")
with open(out, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["split","class","path"])
    for split in ("train","test"):
        base = Path("data")/split
        if not base.exists(): continue
        for cls in sorted([d for d in base.iterdir() if d.is_dir()]):
            for f in (base/cls).glob("*"):
                if f.is_file():
                    writer.writerow([split, cls.name, str(f.resolve())])
print("Manifest saved to", out)

### Compute class weights (to pass to model.fit)

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path

labels = []
for cls in sorted((Path("data")/"train").iterdir()):
    if cls.is_dir():
        n = sum(1 for f in cls.glob("*") if f.is_file())
        labels += [cls.name]*n
classes = np.unique(labels)
cw = compute_class_weight("balanced", classes=classes, y=np.array(labels))
dict(zip(classes, cw))

###  face detection coverage (OpenCV Haar cascades) — roughly measure how many images contain detectable faces (good for face-based pipelines)

In [ ]:
import cv2
from pathlib import Path

detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
count_with_face = 0
total = 0
for f in (Path("data")/"train").rglob("*"):
    if not f.is_file(): continue
    img = cv2.imread(str(f))
    if img is None: continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)
    total += 1
    if len(faces) > 0:
        count_with_face += 1
print("face coverage:", count_with_face, "/", total)

### We focus on 3 emotions for marketing relevance

In [ ]:
TARGET_EMOTIONS = ["happy", "neutral", "sad"]
IMG_SIZE = (48, 48)  # FER2013 standard image size
 
print("=== IPCV: Emotion Detection Project ===")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Target emotions: {TARGET_EMOTIONS}")

## Data Visualization

### 3. CLASS DISTRIBUTION ANALYSIS

In [ ]:
def count_images(base_dir, emotions):
    """Count images per emotion class in a directory."""
    counts = {}
    for emotion in emotions:
        folder = base_dir / emotion
        if folder.exists():
            counts[emotion] = len(list(folder.glob("*.jpg")) + list(folder.glob("*.png")))
        else:
            counts[emotion] = 0
    return counts
 
train_counts = count_images(TRAIN_DIR, TARGET_EMOTIONS)
test_counts  = count_images(TEST_DIR,  TARGET_EMOTIONS)
 
print("\n=== Class Distribution ===")
df_dist = pd.DataFrame({
    "Emotion": TARGET_EMOTIONS,
    "Train":   [train_counts[e] for e in TARGET_EMOTIONS],
    "Test":    [test_counts[e]  for e in TARGET_EMOTIONS],
})
df_dist["Total"] = df_dist["Train"] + df_dist["Test"]
print(df_dist.to_string(index=False))


### Plot class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_dist.set_index("Emotion")[["Train", "Test"]].plot(
    kind="bar", ax=axes[0], color=["#6c5ce7", "#fd79a8"]
)
axes[0].set_title("Class Distribution: Train vs Test")
axes[0].set_xlabel("Emotion")
axes[0].set_ylabel("Number of Images")
axes[0].tick_params(axis="x", rotation=0)
 
df_dist.set_index("Emotion")["Total"].plot(
    kind="pie", ax=axes[1], autopct="%1.1f%%",
    colors=["#55efc4", "#74b9ff", "#ff7675"]
)
axes[1].set_title("Overall Distribution")
axes[1].set_ylabel("")
 
plt.tight_layout()
plt.savefig("../images/01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/01_class_distribution.png")


### 4. SAMPLE IMAGE DISPLAY

In [ ]:
import cv2  # pip install opencv-python
 
def load_sample_images(base_dir, emotions, n=5):
    """Load n sample images per emotion class."""
    samples = {}
    for emotion in emotions:
        folder = base_dir / emotion
        if folder.exists():
            files = list(folder.glob("*.jpg"))[:n]
            images = [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) for f in files]
            samples[emotion] = images
    return samples
 
train_samples = load_sample_images(TRAIN_DIR, TARGET_EMOTIONS, n=5)
 
fig, axes = plt.subplots(len(TARGET_EMOTIONS), 5, figsize=(15, 3 * len(TARGET_EMOTIONS)))
for row, emotion in enumerate(TARGET_EMOTIONS):
    images = train_samples.get(emotion, [])
    for col in range(5):
        ax = axes[row, col]
        if col < len(images) and images[col] is not None:
            ax.imshow(images[col], cmap="gray")
        else:
            ax.axis("off")
        if col == 0:
            ax.set_ylabel(emotion.capitalize(), fontsize=12, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])
 
plt.suptitle("Sample Images per Emotion Class (FER2013 — Train Set)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../images/01_sample_images.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/01_sample_images.png")
 

### 5. IMAGE PROPERTY ANALYSIS

In [ ]:
def analyze_image_properties(base_dir, emotions, sample_size=50):
    """Analyze pixel statistics across a sample of images."""
    stats = []
    for emotion in emotions:
        folder = base_dir / emotion
        if not folder.exists():
            continue
        files = list(folder.glob("*.jpg"))[:sample_size]
        for f in files:
            img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                stats.append({
                    "emotion": emotion,
                    "height": img.shape[0],
                    "width": img.shape[1],
                    "mean_pixel": img.mean(),
                    "std_pixel": img.std(),
                    "min_pixel": img.min(),
                    "max_pixel": img.max(),
                })
    return pd.DataFrame(stats)
 
df_stats = analyze_image_properties(TRAIN_DIR, TARGET_EMOTIONS)
 
print("\n=== Image Pixel Statistics by Emotion ===")
print(df_stats.groupby("emotion")[["mean_pixel", "std_pixel"]].describe().round(2))
 
# Pixel intensity distribution
fig, axes = plt.subplots(1, len(TARGET_EMOTIONS), figsize=(15, 4))
colors = {"happy": "#55efc4", "neutral": "#74b9ff", "sad": "#ff7675"}
for i, emotion in enumerate(TARGET_EMOTIONS):
    subset = df_stats[df_stats["emotion"] == emotion]["mean_pixel"]
    axes[i].hist(subset, bins=20, color=colors.get(emotion, "gray"), edgecolor="white")
    axes[i].set_title(f"{emotion.capitalize()}\n(n={len(subset)})")
    axes[i].set_xlabel("Mean Pixel Intensity")
    axes[i].set_ylabel("Frequency")
 
plt.suptitle("Pixel Intensity Distribution per Emotion", fontsize=13)
plt.tight_layout()
plt.savefig("../images/01_pixel_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/01_pixel_distribution.png")

### 6. SUMMARY FOR REPORT

In [ ]:
print("\n" + "="*50)
print("SUMMARY — copy into your Part 1 / Part 2 report")
print("="*50)
total_train = sum(train_counts.values())
total_test  = sum(test_counts.values())
print(f"Dataset:       FER2013 (filtered to {len(TARGET_EMOTIONS)} classes)")
print(f"Train images:  {total_train}")
print(f"Test images:   {total_test}")
print(f"Image size:    {IMG_SIZE[0]}x{IMG_SIZE[1]} px, grayscale")
print(f"Classes used:  {', '.join(TARGET_EMOTIONS)}")
print(f"\nClass balance:")
for e in TARGET_EMOTIONS:
    pct = train_counts[e] / total_train * 100 if total_train > 0 else 0
    print(f"  {e:<10} {train_counts[e]:>5} train images ({pct:.1f}%)")
print("\n  Note: FER2013 is known to have ~65-68% human labeling accuracy.")
print("   Document this as a dataset limitation in your report.")
